In [0]:
import pandas as pd

In [0]:
%sql
--DROP DATABASE IF EXISTS workspace.silver CASCADE; 

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.silver
COMMENT 'Capa Silver procesados'


In [0]:
#spark.sql("DROP TABLE IF EXISTS workspace.silver.adidas_US_sales")

In [0]:

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.adidas_US_sales (
  id_venta LONG,
  tienda STRING,
  fecha_factura DATE,
  region STRING,
  estado STRING,
  ciudad STRING,
  producto STRING,
  precio_unitario INTEGER,
  unidades_vendidas INTEGER,
  total_ventas INTEGER,
  ganancia_operativa INTEGER,
  canal_venta STRING
)
""")

In [0]:
# Formato a los valores tipo string del df
def formato_strings(valor: str) -> str:
    return valor.strip().upper()

# Limpiar los nombres de las columnas si estas vinieran con espacios o con mayusculas
def limpiar_columnas(columna: str) -> str:
    return columna.strip().lower().replace(" ", "_")


In [0]:
# Leer bronze y pasar a pandas
df_spark = spark.table("workspace.bronze.adidas_US_sales")
df = df_spark.toPandas()

In [0]:
# Limpiar nombres de columnas
df.columns = [limpiar_columnas(col) for col in df.columns]
#Eliminamos la columna "retailer_id"
df = df.drop(columns = ["retailer_id"])

In [0]:
# Renombrar columnas clave
rename_map = {
    "sales_id": "id_venta",
    "retailer":"tienda",
    "invoice_date": "fecha_factura",
    "region": "region",
    "state": "estado",
    "city": "ciudad",
    "product": "producto",
    "price_per_unit": "precio_unitario",
    "units_sold": "unidades_vendidas",
    "total_sales": "total_ventas",
    "operating_profit": "ganancia_operativa",
    "sales_method": "canal_venta"
}

df = df.rename(columns=rename_map, errors="ignore")

In [0]:
# Formato a los valores string
df["tienda"] = df["tienda"].apply(lambda valor: formato_strings(valor))
df["region"] = df["region"].apply(lambda valor: formato_strings(valor))
df["estado"] = df["estado"].apply(lambda valor: formato_strings(valor))
df["ciudad"] = df["ciudad"].apply(lambda valor: formato_strings(valor))
df["producto"] = df["producto"].apply(lambda valor: formato_strings(valor))
df["canal_venta"] = df["canal_venta"].apply(lambda valor: formato_strings(valor))
# Eliminar valores nulos
df = df.dropna()
# Eliminar valores duplicados
df = df.drop_duplicates()

In [0]:
# Volver a Spark el DataFrame
df_spark = spark.createDataFrame(df)

In [0]:
# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.adidas_US_sales")

In [0]:
%sql
--select * from workspace.silver.adidas_us_sales limit 10